# Bokmål / Nynorsk Filtering with SLIDE

Filters Norwegian target sentences using UiO LTG's SLIDE model.
Keeps only Bokmål sentences and writes them to JSONL files.

In [1]:
import json
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import pipeline

d:\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [7]:
# Root is two levels up from current notebook location (data/variant/ → root)
ROOT = Path.cwd().parent.parent

MODEL_NAME = "ltg/SLIDE-base"
DEVICE = 0 if torch.cuda.is_available() else -1

INPUT_FILES = {
    "train": ROOT / "data/final_splits_npd/train.json",
    "val":   ROOT / "data/final_splits_npd/val.json",
    "test":  ROOT / "data/final_splits_npd/test.json",
}

# Filtered output goes here, inside data/variant/
OUT_DIR = ROOT / "data/final_splits_npd_bokmal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Output directory: {OUT_DIR}")
print(f"Input files exist: { {k: v.exists() for k, v in INPUT_FILES.items()} }")

Device: 0
Output directory: d:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\final_splits_npd_bokmal
Input files exist: {'train': True, 'val': True, 'test': True}


## Load SLIDE model

In [8]:
clf = pipeline(
    "text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    top_k=None,
    device=DEVICE,
    trust_remote_code=True,
)

# Quick sanity check
print(clf("Dette er en kort norsk bokmålstekst.")[0])

d:\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\J\.cache\huggingface\hub\models--ltg--SLIDE-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to ef

[{'label': 'nb', 'score': 0.9995701909065247}, {'label': 'sv', 'score': 0.001799131161533296}, {'label': 'da', 'score': 0.001114338985644281}, {'label': 'other', 'score': 0.0007070191204547882}, {'label': 'nn', 'score': 0.000507966848090291}]


## Helper functions

In [9]:
def get_scores(text):
    """Run SLIDE on a text and return {bokmal: float, nynorsk: float}."""
    text = " ".join(str(text).split())
    if not text:
        return {}
    preds = clf(text[:512])[0]
    scores = {}
    for p in preds:
        label = p["label"].lower()
        if any(x in label for x in ["bok", "nob", "nb"]):
            scores["bokmal"] = max(scores.get("bokmal", 0.0), p["score"])
        elif any(x in label for x in ["nyn", "nno", "nn"]):
            scores["nynorsk"] = max(scores.get("nynorsk", 0.0), p["score"])
    return scores


def is_bokmal(scores):
    """Return True if scores confidently indicate Bokmål."""
    return scores.get("bokmal", 0.0) >= 0.80 and scores.get("nynorsk", 0.0) < 0.30

## Sanity check on a few examples

In [10]:
examples = [
    "Dette er en bokmålstekst om petroleumsvirksomhet på norsk sokkel.",
    "Dette er ein nynorsk tekst om petroleumsverksemd på norsk sokkel.",
]

for text in examples:
    scores = get_scores(text)
    print(f"bokmal={is_bokmal(scores)} | scores={scores}")
    print(f"  → {text}\n")

bokmal=True | scores={'bokmal': 0.9994809031486511, 'nynorsk': 0.0004088715650141239}
  → Dette er en bokmålstekst om petroleumsvirksomhet på norsk sokkel.

bokmal=False | scores={'nynorsk': 0.9996986389160156, 'bokmal': 0.0011363724479451776}
  → Dette er ein nynorsk tekst om petroleumsverksemd på norsk sokkel.



## Run filtering on all splits

In [11]:
for split, path in INPUT_FILES.items():
    data = json.loads(path.read_text(encoding="utf-8"))
    out_path = OUT_DIR / f"{split}.jsonl"
    kept = 0

    with out_path.open("w", encoding="utf-8") as f:
        for item in tqdm(data, desc=split):
            scores = get_scores(item["target"])
            if is_bokmal(scores):
                item["bokmal_score"] = round(scores.get("bokmal", 0.0), 4)
                item["nynorsk_score"] = round(scores.get("nynorsk", 0.0), 4)
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                kept += 1

    print(f"{split}: {kept}/{len(data)} kept → {out_path}")

train: 100%|██████████| 13935/13935 [03:14<00:00, 71.57it/s]


train: 10114/13935 kept → d:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\final_splits_npd_bokmal\train.jsonl


val: 100%|██████████| 1737/1737 [00:23<00:00, 72.48it/s]


val: 1305/1737 kept → d:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\final_splits_npd_bokmal\val.jsonl


test: 100%|██████████| 1742/1742 [00:24<00:00, 72.36it/s]

test: 1313/1742 kept → d:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\final_splits_npd_bokmal\test.jsonl
